# LP reduced-form (fallback if IV weak)
Estimate dynamic winner effects using local projections without IV interpretation.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.paths import ANALYSIS_DIR, PAPER_TABLES_DIR
from src.viz_style import set_style

In [2]:
set_style()

panel_path = ANALYSIS_DIR / "rd_event_panel.parquet"
if not panel_path.exists():
    raise FileNotFoundError("Missing event panel. Run 13_construct_efw_shocks_and_outcomes first.")

panel = pd.read_parquet(panel_path)

In [3]:
HORIZONS = [1, 2, 4]
OUTCOMES = {
    "log_gdp_cum": "log_gdp_cum_h{h}",
    "inv_share_avg": "inv_share_avg_h{h}",
    "inflation_path": "inflation_path_h{h}",
}

controls = [
    "lag1_log_gdp_pc_const",
    "lag1_trade_open_gdp",
    "lag1_inflation_cpi_ann_pct",
    "lag1_efw_summary",
]

def safe_ols(frame: pd.DataFrame, y: str, x: list[str], cluster: str | None = None):
    needed = [y] + x
    frame = frame.dropna(subset=needed).copy()
    if frame.empty:
        return np.nan, np.nan, np.nan, 0
    X = sm.add_constant(frame[x])
    model = sm.OLS(frame[y], X)
    if cluster and cluster in frame.columns:
        groups = frame[cluster]
        if groups.nunique(dropna=True) < 2 or len(frame) <= X.shape[1]:
            result = model.fit(cov_type="HC1")
        else:
            try:
                result = model.fit(cov_type="cluster", cov_kwds={"groups": groups})
            except (ValueError, ZeroDivisionError):
                result = model.fit(cov_type="HC1")
    else:
        result = model.fit(cov_type="HC1")
    coef = float(result.params.get("winner_market", np.nan))
    se = float(result.bse.get("winner_market", np.nan))
    pval = float(result.pvalues.get("winner_market", np.nan))
    return coef, se, pval, int(result.nobs)

In [4]:
rows = []
for outcome, template in OUTCOMES.items():
    for h in HORIZONS:
        y_col = template.format(h=h)
        coef, se, pval, n_obs = safe_ols(
            panel,
            y=y_col,
            x=["winner_market"] + controls,
            cluster="iso3c",
        )
        rows.append(
            {
                "outcome": outcome,
                "horizon": h,
                "coef": coef,
                "se": se,
                "pvalue": pval,
                "n_obs": n_obs,
                "method": "lp_ols",
            }
        )

lp_df = pd.DataFrame(rows)
PAPER_TABLES_DIR.mkdir(parents=True, exist_ok=True)
output_path = PAPER_TABLES_DIR / "lp_reduced_form.csv"
lp_df.to_csv(output_path, index=False)

In [5]:
display(lp_df.style.set_caption("LP reduced-form (winner effects)"))

,outcome,horizon,coef,se,pvalue,n_obs,method
0,log_gdp_cum,1,0.011305,0.009154,0.216870,200,lp_ols
1,log_gdp_cum,2,0.011869,0.011739,0.311984,200,lp_ols
2,log_gdp_cum,4,0.024318,0.015122,0.107800,190,lp_ols
3,inv_share_avg,1,1.473006,0.489951,0.002643,200,lp_ols
4,inv_share_avg,2,1.326694,0.478045,0.005516,200,lp_ols
5,inv_share_avg,4,1.561781,0.506958,0.002065,200,lp_ols
6,inflation_path,1,0.695149,0.378693,0.066409,200,lp_ols
7,inflation_path,2,0.698458,0.368592,0.058101,200,lp_ols
8,inflation_path,4,0.569964,0.548856,0.299057,190,lp_ols


## Interpretation
Local projection estimates summarize dynamic winner effects without IV interpretation,
providing a descriptive fallback when RD‑IV is weak.